# FER-2013 — High Accuracy Training on Google Colab

**Before running:**
1. Runtime → Change runtime type → **GPU (T4)**
2. Upload your `dataset/` folder to Google Drive
3. Run all cells in order
4. Download `emotion_model_colab.h5` and rename to `emotion_model.h5` in your local `model/` folder

In [ ]:
# ── Step 1: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Check GPU is available
import tensorflow as tf
print('GPU available:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ── Step 2: Set your dataset path in Drive ───────────────────────────────────
import os

# CHANGE THIS to wherever you uploaded the dataset folder in your Drive
DRIVE_DATASET = '/content/drive/MyDrive/MajorProject/dataset'

TRAIN_DIR = os.path.join(DRIVE_DATASET, 'train')
TEST_DIR  = os.path.join(DRIVE_DATASET, 'test')

print('Train dir exists:', os.path.exists(TRAIN_DIR))
print('Test  dir exists:', os.path.exists(TEST_DIR))

# Count images per class
for cls in sorted(os.listdir(TRAIN_DIR)):
    n = len(os.listdir(os.path.join(TRAIN_DIR, cls)))
    print(f'  {cls:10s}: {n} images')

In [ ]:
# ── Step 3: Install / import libraries ──────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, BatchNormalization, MaxPooling2D,
    Dropout, Flatten, Dense, GlobalAveragePooling2D
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

print('TensorFlow version:', tf.__version__)

In [ ]:
# ── Step 4: Config ───────────────────────────────────────────────────────────
IMG_SIZE    = 48
BATCH_SIZE  = 128   # larger batch = faster on GPU
EPOCHS      = 80    # early stopping will kick in before this
LR          = 1e-3
MODEL_SAVE  = 'emotion_model_colab.h5'

EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
NUM_CLASSES = len(EMOTIONS)

In [ ]:
# ── Step 5: Data generators with stronger augmentation ──────────────────────
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.15,
    shear_range=0.1,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
)

test_datagen = ImageDataGenerator(rescale=1.0/255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale', batch_size=BATCH_SIZE,
    class_mode='categorical', classes=EMOTIONS, shuffle=True
)

test_gen = test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale', batch_size=BATCH_SIZE,
    class_mode='categorical', classes=EMOTIONS, shuffle=False
)

print(f'Train: {train_gen.samples} | Test: {test_gen.samples}')

In [ ]:
# ── Step 6: Compute class weights (fix imbalance — disgust has far fewer samples) ──
labels = train_gen.classes
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
class_weight_dict = dict(enumerate(class_weights))
print('Class weights:', {EMOTIONS[k]: round(v, 2) for k, v in class_weight_dict.items()})

In [ ]:
# ── Step 7: Full 4-block CNN (runs fast on Colab GPU) ───────────────────────
def build_model():
    model = Sequential(name='EmotionCNN_HighAcc')

    # Block 1
    model.add(Conv2D(64, (3,3), padding='same', activation='relu',
                     input_shape=(48, 48, 1)))
    model.add(BatchNormalization())
    model.add(Conv2D(64, (3,3), padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(2, 2))
    model.add(Dropout(0.25))

    # Block 2
    model.add(Conv2D(128, (3,3), padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(Conv2D(128, (3,3), padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(2, 2))
    model.add(Dropout(0.25))

    # Block 3
    model.add(Conv2D(256, (3,3), padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(Conv2D(256, (3,3), padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(2, 2))
    model.add(Dropout(0.25))

    # Block 4
    model.add(Conv2D(512, (3,3), padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(Conv2D(512, (3,3), padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(2, 2))
    model.add(Dropout(0.25))

    # Head
    model.add(Flatten())
    model.add(Dense(512, activation='relu', kernel_regularizer=l2(0.001)))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))
    model.add(Dense(256, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(NUM_CLASSES, activation='softmax'))

    return model

model = build_model()
model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# ── Step 8: Callbacks ────────────────────────────────────────────────────────
callbacks = [
    ModelCheckpoint(MODEL_SAVE, monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=5, min_lr=1e-7, verbose=1),
]

In [ ]:
# ── Step 9: Train ────────────────────────────────────────────────────────────
history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=test_gen,
    class_weight=class_weight_dict,   # handles class imbalance
    callbacks=callbacks,
)

In [ ]:
# ── Step 10: Evaluate ────────────────────────────────────────────────────────
loss, acc = model.evaluate(test_gen, verbose=1)
print(f'\nTest Loss     : {loss:.4f}')
print(f'Test Accuracy : {acc*100:.2f}%')

test_gen.reset()
preds  = model.predict(test_gen, verbose=1)
y_pred = np.argmax(preds, axis=1)
y_true = test_gen.classes

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=EMOTIONS))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9,7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=EMOTIONS, yticklabels=EMOTIONS)
plt.title('Confusion Matrix — FER-2013 (Colab)')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()

In [ ]:
# ── Step 11: Training curves ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy'); axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── Step 12: Download the trained model ──────────────────────────────────────
from google.colab import files
files.download(MODEL_SAVE)
print(f'Downloaded {MODEL_SAVE}')
print('Rename it to emotion_model.h5 and place in your local model/ folder.')